In [1]:
# Cell 1 — Install dependencies
!pip install -q transformers peft accelerate huggingface_hub
!pip install -q langchain langchain-community sentence-transformers faiss-cpu
!pip install -q gradio
print('✅ All packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
✅ All packages installed!


In [2]:
# Cell 2 — Load API keys and login
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

ADZUNA_APP_ID  = userdata.get('ADZUNA_APP_ID')
ADZUNA_APP_KEY = userdata.get('ADZUNA_APP_KEY')

print('✅ HuggingFace logged in!')
print(f'Adzuna App ID: {ADZUNA_APP_ID[:4]}****')

✅ HuggingFace logged in!
Adzuna App ID: 53ee****


In [3]:
# Cell 3 — Load FAISS index and chunks from Google Drive
from google.colab import drive
import faiss
import pickle

drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/job_market_navigator'

index = faiss.read_index(f'{save_dir}/faiss_index.bin')
with open(f'{save_dir}/chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)

print(f'✅ FAISS index loaded — vectors: {index.ntotal}')
print(f'✅ Chunks loaded — total: {len(chunks)}')

Mounted at /content/drive
✅ FAISS index loaded — vectors: 300
✅ Chunks loaded — total: 300


In [4]:
# Cell 4 — Load fine-tuned model from HuggingFace Hub
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL = 'distilgpt2'
FINETUNED_MODEL = 'kashanikram/job-market-navigator-distilgpt2'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto'
)

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL)
model.eval()

print('✅ Fine-tuned model loaded!')

Loading tokenizer...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading LoRA adapter...


adapter_config.json:   0%|          | 0.00/947 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/591k [00:00<?, ?B/s]

✅ Fine-tuned model loaded!


In [5]:
# Cell 5 — Load embedding model for RAG
from sentence_transformers import SentenceTransformer
import numpy as np

print('Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Embedding model loaded!')

def search_rag(query, top_k=3):
    query_embedding = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    results = [chunks[i] for i in indices[0]]
    return '\n\n'.join(results)

def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=400
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = full.split('### Response:')[-1].split('### End')[0].strip()
    return response

print('✅ RAG search and generator ready!')

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!
✅ RAG search and generator ready!


In [6]:
# Cell 6 — Agentic Router (decides RAG vs Fine-tuned model)
def agent_router(query):
    """
    Agent decides which tool to use:
    - RAG: current job data, salaries, companies, locations
    - Fine-tuned model: career advice, skills, requirements
    - Both: complex queries needing live data + deep knowledge
    """
    query_lower = query.lower()

    rag_keywords = [
        'salary', 'company', 'location', 'toronto', 'vancouver',
        'canada', 'hiring', 'job opening', 'current', 'latest',
        'remote', 'hybrid', 'today', 'now', 'available'
    ]

    model_keywords = [
        'skills', 'requirements', 'how to', 'advice', 'career',
        'learn', 'prepare', 'resume', 'interview', 'qualify',
        'experience needed', 'what should'
    ]

    rag_score   = sum(1 for k in rag_keywords if k in query_lower)
    model_score = sum(1 for k in model_keywords if k in query_lower)

    if rag_score > model_score:
        return 'rag'
    elif model_score > rag_score:
        return 'model'
    else:
        return 'both'

def run_agent(query):
    route = agent_router(query)
    print(f'[Agent] Query: "{query}"')
    print(f'[Agent] Route: {route.upper()}')

    if route == 'rag':
        context = search_rag(query)
        prompt = (
            f"### Instruction:\n{query}\n\n"
            f"Context from live job data:\n{context[:600]}\n\n"
            f"### Response:\n"
        )
        answer = generate_answer(prompt)

    elif route == 'model':
        prompt = (
            f"### Instruction:\n{query}\n\n"
            f"### Response:\n"
        )
        answer = generate_answer(prompt)

    else:  # both
        context = search_rag(query)
        prompt = (
            f"### Instruction:\n{query}\n\n"
            f"Context:\n{context[:400]}\n\n"
            f"### Response:\n"
        )
        answer = generate_answer(prompt)

    return route, answer

# Test agent
print('=== Agent Test ===\n')
queries = [
    'What is the salary for data scientist in Toronto?',
    'What skills do I need for machine learning engineer?',
    'Which companies are hiring AI engineers in Canada right now?'
]

for q in queries:
    route, answer = run_agent(q)
    print(f'Answer: {answer[:200]}')
    print('-' * 50)

=== Agent Test ===

[Agent] Query: "What is the salary for data scientist in Toronto?"
[Agent] Route: RAG
Answer: ### Response
--------------------------------------------------
[Agent] Query: "What skills do I need for machine learning engineer?"
[Agent] Route: MODEL
Answer: "Core Responsibilities": "A computer science graduate student with a passion for machine learning and machine learning and machine learning, and a computer science graduate student with a passion for 
--------------------------------------------------
[Agent] Query: "Which companies are hiring AI engineers in Canada right now?"
[Agent] Route: RAG
Answer: ### Responses:
### Responses:
### Responses:
### Responses:
### Responses:
### Responses:
### Responses:
### Responses:
### Responses:
### Responses:
### Responses:
### Responses:
### Responses:
### R
--------------------------------------------------


In [8]:
# Cell 6 — Fixed Agent Router + Better Generation
def agent_router(query):
    query_lower = query.lower()

    rag_keywords = [
        'salary', 'company', 'location', 'toronto', 'vancouver',
        'canada', 'hiring', 'job opening', 'current', 'latest',
        'remote', 'hybrid', 'today', 'now', 'available'
    ]

    model_keywords = [
        'skills', 'requirements', 'how to', 'advice', 'career',
        'learn', 'prepare', 'resume', 'interview', 'qualify',
        'experience needed', 'what should'
    ]

    rag_score   = sum(1 for k in rag_keywords if k in query_lower)
    model_score = sum(1 for k in model_keywords if k in query_lower)

    if rag_score > model_score:
        return 'rag'
    elif model_score > rag_score:
        return 'model'
    else:
        return 'both'

def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=350
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,          # Greedy — more stable output
            repetition_penalty=1.5,   # Stops repetition
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Clean output
    if '### Response:' in full:
        response = full.split('### Response:')[-1]
    else:
        response = full[len(prompt):]
    response = response.split('### End')[0].strip()
    response = response.split('### Instruction')[0].strip()
    return response if response else 'No answer generated.'

def format_rag_answer(query, chunks_text):
    """Extract direct info from RAG chunks instead of generating"""
    lines = []
    for chunk in chunks_text.split('---'):
        if not chunk.strip():
            continue
        info = {}
        for line in chunk.strip().split('\n'):
            if ':' in line:
                key, _, val = line.partition(':')
                info[key.strip()] = val.strip()
        if info:
            title    = info.get('Job Title', '')
            company  = info.get('Company', '')
            location = info.get('Location', '')
            salary   = info.get('Salary', 'Not specified')
            desc     = info.get('Description', '')[:150]
            if title:
                lines.append(f"• {title} at {company} ({location}) — Salary: {salary}\n  {desc}")
    return '\n\n'.join(lines[:3]) if lines else 'No results found.'

def run_agent(query):
    route = agent_router(query)

    if route == 'rag' or route == 'both':
        context = search_rag(query, top_k=3)
        answer = format_rag_answer(query, context)
    else:
        prompt = f"### Instruction:\n{query}\n\n### Response:\n"
        answer = generate_answer(prompt)

    return route, answer

# Test
print('=== Agent Test ===\n')
queries = [
    'What is the salary for data scientist in Toronto?',
    'What skills do I need for machine learning engineer?',
    'Which companies are hiring AI engineers in Canada right now?'
]

for q in queries:
    route, answer = run_agent(q)
    print(f'Q: {q}')
    print(f'Route: {route.upper()}')
    print(f'A: {answer}')
    print('-' * 50)

=== Agent Test ===

Q: What is the salary for data scientist in Toronto?
Route: RAG
A: • Data Scientist at Larus Technologies (Ottawa, Ottawa region) — Salary: 100000
  Full-time: 37.5 Hours per week, Monday to Friday Location: Ottawa Office (3 days per week / hybrid), 170 Laurier Ave West, Suite 310, Ottawa, ON K1P 5

• Data Scientist at Larus Technologies (Ottawa, Ottawa region) — Salary: Not specified
  Full-time: 37.5 Hours per week, Monday to Friday Location: Ottawa Office (3 days per week / hybrid), 170 Laurier Ave West, Suite 310, Ottawa, ON K1P 5

• Data Scientist at Larus Technologies (Ottawa, Ottawa region) — Salary: Not specified
  Job Description Job Description Full-time: 37.5 Hours per week, Monday to Friday Location: Ottawa Office (3 days per week / hybrid), 170 Laurier Ave W
--------------------------------------------------
Q: What skills do I need for machine learning engineer?
Route: MODEL
A: "Core Responsibilities": {
 }
  End : "Main role of Machine Learning Engine

In [9]:
# Cell 7 — Gradio Demo
import gradio as gr

def chat(query):
    if not query.strip():
        return 'Please enter a question!'

    route, answer = run_agent(query)
    route_label = {
        'rag': 'Live Job Data (RAG)',
        'model': 'AI Model',
        'both': 'RAG + AI Model'
    }.get(route, route)

    return f"[Source: {route_label}]\n\n{answer}"

demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(
        placeholder='e.g. Which companies are hiring AI engineers in Canada?',
        label='Ask about the job market',
        lines=2
    ),
    outputs=gr.Textbox(
        label='Answer',
        lines=8
    ),
    title='AI Job Market Navigator',
    description='Ask about Canadian tech jobs, salaries, companies hiring, and skills needed.',
    examples=[
        ['Which companies are hiring AI engineers in Canada right now?'],
        ['What is the salary for data scientist in Toronto?'],
        ['What skills do I need for machine learning engineer?'],
        ['Which companies are hiring software engineers in Vancouver?'],
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://045b9d95f59bfeb5ff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
